In [0]:
content='''MARKET FOR REGISTRANT’S COMMON EQUITY, RELATED STOCKHOLDER MATTERS, AND ISSUER PURCHASES OF EQUITY SECURITIES As of October 2, 2015, Alphabet Inc. became the successor issuer of Google Inc. pursuant to Rule 12g-3(a) under the Exchange Act. Our Class A stock has been listed on the Nasdaq Global Select Market under the symbol “GOOG” since August 19, 2004, and under the symbol "GOOGL" since April 3, 2014. Prior to August 19, 2004, there was no public market for our stock. Our Class B stock is neither listed nor traded. Our Class C stock has been listed on the Nasdaq Global Select Market under the symbol “GOOG” since April 3, 2014. Holders of Record As of December 31, 2025 , there were approxim a tely 5,861 and 1,611 stock holders of record of our Class A stock and Class C stock, respectively. Because many of our shares of Class A stock and Class C stock are held by brokers and other institutions on behalf of stockholders, we are unable to estimate the total number of stockholders represented by these record holders. As of December 31, 2025, there were approximately 57 st ockholders of record of our Class B stock. Dividend Program Dividend payments to stockholders of Class A, Class B, and Class C shares were $4.8 billion, $703 million, and $4.5 billion, respectively, for the year ended December 31, 2025. The company intends to pay quarterly cash dividends in the future, subject to review and approval by the company’s Board of Directors in its sole discretion. We regularly evaluate our cash and capital structure, including the size, pace, and form of capital return to stockholders. Issuer Purchases of Equity Securities The following table presents information with respect to Alphabet's repurchases of Class A and Class C stock during the quarter ended December 31, 2025 : | | | | | | | | | | | | | | | | | | | | | | | | | | | | | | | | | | | | | | | | | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | | Period | | Total Number of Class A Shares Purchased (in thousands) (1) | | Total Number of Class C Shares Purchased (in thousands) (1) | | Average Price Paid per Class A Share (2) | | Average Price Paid per Class C Share (2) | | Total Number of Shares Purchased as Part of Publicly Announced Programs (in thousands) (1) | | Approximate Dollar Value of Shares that May Yet Be Purchased Under the Program (in millions) | | October 1 - 31 | | 841 | | | 13,305 | | | $ | 253.20 | | | $ | 255.03 | | | 14,146 | | | $ | 71,231 | | | November 1 - 30 | | 209 | | | 5,885 | | | $ | 286.00 | | | $ | 281.46 | | | 6,094 | | | $ | 69,503 | | | December 1 - 31 | | 0 | | | 0 | | | $ | 0.00 | | | $ | 0.00 | | | 0 | | | $ | 69,503 | | | Total | | 1,050 | | | 19,190 | | | | | | | 20,240 | | | | (1) In April 2024, the company's Board of Directors authorized a $70.0 billion share repurchase program for its Class A and Class C shares. In April 2025, the company's Board of Directors authorized an additional $70.0 billion share repurchase program for its Class A and Class C shares. Repurchases are being executed from time to time, subject to general business and market conditions and other investment opportunities, through open market purchases or privately negotiated transactions, including through Rule 10b5-1 plans. The repurchase programs do not have an expiration date. For additional information related to share repurchases, see Note 11 o f the Notes to Consolidated Financial Statements included in Item 8 of this Annual Report on Form 10-K. (2) Average price paid per share includes costs associated with the repurchases. 25. | | | | | | | | --- | --- | --- | --- | --- | --- | | Table of Contents | Alphabet Inc. | Stock Performance Graphs The graph below matches Alphabet Inc. Class A's cumulative five-year total stockholder return on common stock with the cumulative total returns of the S&P 500 index, the NASDAQ Composite index, and the RDG Internet Composite index. The graph tracks the performance of a $100 investment in our common stock and in each index (with the reinvestment of all dividends) from December 31, 2020, to December 31, 2025. The returns shown are based on historical results and are not intended to suggest future performance. COMPARISON OF CUMULATIVE 5-YEAR TOTAL RETURN* ALPHABET INC. CLASS A COMMON STOCK Among Alphabet Inc., the S&P 500 Index, the NASDAQ Composite Index, and the RDG Internet Composite Index *$100 invested on December 31, 2020, in stock or index, including reinvestment of dividends. Copyright © 2026 S&P, a division of The McGraw-Hill Companies Inc. All rights reserved. 26. | | | | | | | | --- | --- | --- | --- | --- | --- | | Table of Contents | Alphabet Inc. | The graph below matches Alphabet Inc. Class C's cumulative five-year total stockholder return on capital stock with the cumulative total returns of the S&P 500 index, the NASDAQ Composite index, and the RDG Internet Composite index. The graph tracks the performance of a $100 investment in our Class C capital stock and in each index (with the reinvestment of all dividends) from December 31, 2020, to December 31, 2025. The returns shown are based on historical results and are not intended to suggest future performance. COMPARISON OF CUMULATIVE 5-YEAR TOTAL RETURN* ALPHABET INC. CLASS C CAPITAL STOCK Among Alphabet Inc., the S&P 500 Index, the NASDAQ Composite Index, and the RDG Internet Composite Index *$100 invested on December 31, 2020, in stock or in index, including reinvestment of dividends. Copyright © 2026 S&P, a division of The McGraw-Hill Companies Inc. All rights reserved.'''




for i, line in enumerate(content.splitlines()):
    print(f"{i:03d}: {repr(line)}")

In [0]:
import re
import hashlib
from typing import List, Dict


# ============================================================
# CONFIG
# ============================================================

TARGET_CHARS = 3500       # soft target for normal chunks
MAX_CHARS = 5000          # hard maximum where possible
MIN_TEXT_CHARS = 100      # ignore tiny text fragments


# ============================================================
# HELPERS
# ============================================================

def normalize_text(text: str) -> str:
    """
    Light normalization only.

    Important:
    We do NOT aggressively normalize whitespace because the
    source contains extraction artifacts such as:
        approxim a tely
        st ockholders
    """

    text = text.replace("\r", " ")
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()


def is_separator_cell(cell: str) -> bool:
    """
    Markdown table separator cell:
        ---
        :---
        ---:
        :---:
    """
    cell = cell.strip()

    return bool(
        re.fullmatch(r":?-{3,}:?", cell)
    )


def split_pipe_cells(text: str) -> List[str]:
    """
    Extract pipe-delimited cells from a flattened Markdown table.

    Example:
        | A | B | C |

    -> ['A', 'B', 'C']
    """

    text = text.strip()

    if not text.startswith("|"):
        return []

    # Remove outer pipe
    if text.endswith("|"):
        text = text[1:-1]
    else:
        text = text[1:]

    return text.split("|")


def find_separator_start(text: str):
    """
    Find a Markdown separator row inside the flattened document.

    We are looking for something like:

        | --- | --- | --- |

    Since the entire Silver content may be one line, we cannot
    depend on newline boundaries.
    """

    pattern = re.compile(
        r"""
        \|
        (?:
            \s*
            :?-{3,}:?
            \s*
            \|
        ){2,}
        """,
        re.VERBOSE
    )

    return pattern.search(text)


def find_table_start(text: str, separator_start: int) -> int:
    """
    Walk backwards from the separator row to find the beginning
    of the pipe-based table.

    We stop when we encounter meaningful non-table text.
    """

    # Search backwards for a reasonable pipe-run.
    prefix = text[:separator_start]

    # Find the last substantial text boundary before the table.
    matches = list(re.finditer(r"\|", prefix))

    if not matches:
        return separator_start

    # Start from the last pipe before separator and walk backwards.
    pipe_positions = [m.start() for m in matches]

    start = pipe_positions[-1]

    # Move backwards through pipe-delimited content.
    while start > 0:

        previous_pipe = text.rfind("|", 0, start)

        if previous_pipe == -1:
            break

        between = text[previous_pipe + 1:start]

        # If the content between pipes contains obvious sentence text
        # with a reasonable amount of characters, continue treating it
        # as part of the table header.
        start = previous_pipe

        # Don't accidentally consume the entire document.
        if start == 0:
            break

    return start


def extract_table(text: str, separator_match):
    """
    Extract a complete table from a flattened string.

    The separator row tells us that we are inside a table.
    We then consume the pipe-based region until normal text resumes.
    """

    sep_start = separator_match.start()
    sep_end = separator_match.end()

    # --------------------------------------------------------
    # Find table beginning
    # --------------------------------------------------------

    before_separator = text[:sep_start]

    # Find the beginning of the current pipe-run.
    last_text_boundary = max(
        before_separator.rfind(". "),
        before_separator.rfind(": "),
        before_separator.rfind("? "),
        before_separator.rfind("! "),
        before_separator.rfind(") "),
    )

    candidate_start = before_separator.rfind("|", 0, sep_start)

    if candidate_start == -1:
        return None

    # We need to include the header cells before separator.
    # Walk backwards while the text remains pipe/table-like.
    start = candidate_start

    while start > 0:

        previous_pipe = text.rfind("|", 0, start)

        if previous_pipe == -1:
            break

        between = text[previous_pipe + 1:start].strip()

        # If this section looks like ordinary prose, stop.
        if between and len(between) > 100:
            break

        start = previous_pipe

    # --------------------------------------------------------
    # Determine number of columns from separator row
    # --------------------------------------------------------

    separator_text = separator_match.group(0)
    separator_cells = split_pipe_cells(separator_text)

    if len(separator_cells) < 2:
        return None

    column_count = len(separator_cells)

    # --------------------------------------------------------
    # Find end of table
    # --------------------------------------------------------

    # Everything after separator.
    remainder = text[sep_end:]

    # We consume pipe-delimited cells.
    #
    # Because the original content has been flattened, we cannot
    # rely on newline boundaries.
    #
    # Instead we find the next obvious non-table text boundary.

    pipe_matches = list(re.finditer(r"\|", remainder))

    if not pipe_matches:
        end = len(text)
        return text[start:end], end

    # Find a region where normal prose starts.
    end = sep_end

    # Look for patterns such as:
    #   | ... | | ... | (1) In April...
    #
    # We consider a pipe-run followed by normal text as table end.

    boundary_pattern = re.compile(
        r"\|\s*(?=[A-Za-z(])"
    )

    # More practical approach:
    # search for a period/footnote followed by text outside pipes.

    candidates = re.finditer(
        r"""
        \|
        \s*
        (?:
            \(\d+\)
            |
            [A-Z][a-z]{2,}
        )
        """,
        remainder,
        re.VERBOSE
    )

    for match in candidates:
        # This could still be a cell, so inspect the surrounding
        # text. We only use it as an end if there is a substantial
        # text segment after it.
        after = remainder[match.start():]

        if len(after) > 80:
            end = sep_end + match.start()
            break

    else:
        # If no obvious prose boundary was found, table extends
        # to the end.
        end = len(text)

    table_text = text[start:end].strip()

    return table_text, end


# ============================================================
# STRUCTURAL BLOCK EXTRACTION
# ============================================================

def extract_blocks(content: str) -> List[Dict]:
    """
    Convert flattened Silver content into structural blocks.

    Output:

        [
            {
                "block_index": 0,
                "block_type": "text",
                "block_text": "..."
            },
            {
                "block_index": 1,
                "block_type": "table",
                "block_text": "| ... |"
            }
        ]

    Important:
    We deliberately call non-table regions "text", NOT "paragraph",
    because your Silver representation has lost reliable paragraph
    boundaries.
    """

    content = content.strip()

    blocks = []

    cursor = 0
    block_index = 0

    while cursor < len(content):

        remaining = content[cursor:]

        separator_match = find_separator_start(remaining)

        # ----------------------------------------------------
        # No more tables
        # ----------------------------------------------------

        if not separator_match:

            text = normalize_text(remaining)

            if len(text) >= MIN_TEXT_CHARS:
                blocks.append({
                    "block_index": block_index,
                    "block_type": "text",
                    "block_text": text
                })

            break

        # ----------------------------------------------------
        # Text before table
        # ----------------------------------------------------

        table_relative_start = separator_match.start()

        text_before = remaining[:table_relative_start]

        text_before = normalize_text(text_before)

        if len(text_before) >= MIN_TEXT_CHARS:

            blocks.append({
                "block_index": block_index,
                "block_type": "text",
                "block_text": text_before
            })

            block_index += 1

        # ----------------------------------------------------
        # Extract table
        # ----------------------------------------------------

        result = extract_table(
            remaining,
            separator_match
        )

        if result is None:
            # Safety fallback
            text = normalize_text(remaining)

            if len(text) >= MIN_TEXT_CHARS:
                blocks.append({
                    "block_index": block_index,
                    "block_type": "text",
                    "block_text": text
                })

            break

        table_text, table_end = result

        table_text = table_text.strip()

        if table_text:

            blocks.append({
                "block_index": block_index,
                "block_type": "table",
                "block_text": table_text
            })

            block_index += 1

        # Move cursor
        cursor += table_end

    return blocks


# ============================================================
# TEXT CHUNKING
# ============================================================

def split_text_into_units(text: str) -> List[str]:
    """
    Split text into reasonably semantic units.

    Since Silver lost paragraph boundaries, use sentence-ish
    boundaries rather than pretending we have paragraphs.
    """

    text = normalize_text(text)

    # Split after sentence punctuation.
    units = re.split(
        r"(?<=[.!?])\s+(?=[A-Z(])",
        text
    )

    units = [
        unit.strip()
        for unit in units
        if unit.strip()
    ]

    return units


def split_large_text(text: str, max_chars: int) -> List[str]:
    """
    Safety fallback for very large text blocks.

    Prefer sentence boundaries, then hard character splitting.
    """

    units = split_text_into_units(text)

    chunks = []
    current = ""

    for unit in units:

        if len(unit) > max_chars:

            if current:
                chunks.append(current.strip())
                current = ""

            # Hard split only when a single sentence is too large.
            for i in range(0, len(unit), max_chars):
                chunks.append(
                    unit[i:i + max_chars].strip()
                )

            continue

        candidate = (
            unit
            if not current
            else current + " " + unit
        )

        if len(candidate) <= max_chars:
            current = candidate

        else:
            if current:
                chunks.append(current.strip())

            current = unit

    if current:
        chunks.append(current.strip())

    return chunks


# ============================================================
# FINAL CHUNK CONSTRUCTION
# ============================================================

def build_chunks(
    blocks: List[Dict],
    target_chars: int = TARGET_CHARS,
    max_chars: int = MAX_CHARS
) -> List[Dict]:
    """
    Build RAG chunks from structural blocks.

    Rules:

    1. Tables remain atomic whenever possible.
    2. Text blocks can be combined.
    3. We don't combine across a table.
    4. Very large text blocks are sentence-split.
    5. Very large tables are split separately.
    """

    chunks = []

    current_parts = []
    current_length = 0

    def flush():

        nonlocal current_parts, current_length

        if not current_parts:
            return

        chunks.append({
            "chunk_index": len(chunks),
            "chunk_type": "text",
            "chunk_text": "\n\n".join(current_parts)
        })

        current_parts = []
        current_length = 0

    for block in blocks:

        block_type = block["block_type"]
        block_text = block["block_text"]

        # ====================================================
        # TABLE
        # ====================================================

        if block_type == "table":

            # Never combine text + table into an oversized chunk.
            if current_parts:
                flush()

            # Normal table fits.
            if len(block_text) <= max_chars:

                chunks.append({
                    "chunk_index": len(chunks),
                    "chunk_type": "table",
                    "chunk_text": block_text
                })

            else:
                # Large table needs special handling.
                table_chunks = split_large_table(
                    block_text,
                    max_chars=max_chars
                )

                for table_chunk in table_chunks:

                    chunks.append({
                        "chunk_index": len(chunks),
                        "chunk_type": "table",
                        "chunk_text": table_chunk
                    })

            continue

        # ====================================================
        # TEXT
        # ====================================================

        units = split_text_into_units(block_text)

        for unit in units:

            if len(unit) > max_chars:

                # Flush current first
                flush()

                large_parts = split_large_text(
                    unit,
                    max_chars
                )

                for part in large_parts:

                    chunks.append({
                        "chunk_index": len(chunks),
                        "chunk_type": "text",
                        "chunk_text": part
                    })

                continue

            candidate = (
                unit
                if not current_parts
                else "\n\n".join(current_parts + [unit])
            )

            # Soft target reached.
            if (
                current_parts
                and len(candidate) > target_chars
            ):
                flush()

            current_parts.append(unit)

            current_length = len(
                "\n\n".join(current_parts)
            )

    flush()

    return chunks


# ============================================================
# LARGE TABLE SPLITTING
# ============================================================

def split_large_table(
    table_text: str,
    max_chars: int
) -> List[str]:
    """
    Split an oversized Markdown table by rows.

    The header/separator is repeated in every resulting chunk.

    This is a safety mechanism only.
    Normally we want tables to remain whole.
    """

    lines = table_text.splitlines()

    if len(lines) < 3:
        # No reliable row structure.
        return [
            table_text[i:i + max_chars]
            for i in range(0, len(table_text), max_chars)
        ]

    header = lines[:2]
    rows = lines[2:]

    chunks = []

    current = header.copy()
    current_length = sum(len(x) + 1 for x in current)

    for row in rows:

        if (
            current_length + len(row) + 1
            > max_chars
            and len(current) > 2
        ):

            chunks.append(
                "\n".join(current)
            )

            current = header.copy()
            current_length = sum(
                len(x) + 1
                for x in current
            )

        current.append(row)
        current_length += len(row) + 1

    if len(current) > 2:

        chunks.append(
            "\n".join(current)
        )

    return chunks


# ============================================================
# DETERMINISTIC CHUNK ID
# ============================================================

def create_chunk_id(
    document_id: str,
    section_id: str,
    chunk_index: int,
    chunk_text: str
) -> str:

    raw = (
        f"{document_id}|"
        f"{section_id}|"
        f"{chunk_index}|"
        f"{chunk_text}"
    )

    return hashlib.sha256(
        raw.encode("utf-8")
    ).hexdigest()


# ============================================================
# COMPLETE DOCUMENT FUNCTION
# ============================================================

def chunk_document(
    content: str,
    document_id: str,
    section_id: str,
    target_chars: int = TARGET_CHARS,
    max_chars: int = MAX_CHARS
) -> List[Dict]:

    blocks = extract_blocks(content)

    chunks = build_chunks(
        blocks,
        target_chars=target_chars,
        max_chars=max_chars
    )

    final = []

    for chunk in chunks:

        chunk_index = chunk["chunk_index"]

        final.append({
            "document_id": document_id,
            "section_id": section_id,
            "chunk_id": create_chunk_id(
                document_id,
                section_id,
                chunk_index,
                chunk["chunk_text"]
            ),
            "chunk_index": chunk_index,
            "chunk_type": chunk["chunk_type"],
            "chunk_text": chunk["chunk_text"],
            "char_count": len(chunk["chunk_text"])
        })

    return final

In [0]:
from pyspark.sql.types import *

gold_schema = StructType([
    StructField("company_name", StringType(), True),
    StructField("cik", StringType(), True),
    StructField("filling_date", StringType(), True),
    StructField("accession_number", StringType(), True),
    StructField("filling_type", StringType(), True),
    StructField("doc_id",StringType(),True),
    StructField("sec_id",StringType(),True),
    StructField("chunk_id",StringType(),True),
    StructField("chunk_index",IntegerType(),True),
    StructField("chunk_type",StringType(),True),
    StructField("chunk_text",StringType(),True),
    StructField("char_count",IntegerType(),True),
    StructField("processing_timestamp",StringType(),True),
    StructField("sec_number",StringType(),True),
    StructField("sec_title",StringType(),True),
    StructField("file_name",StringType(),True)
])

In [0]:
from pyspark import SparkSession


#spark session for gold 
spark=SparkSession.builder.appName("Gold").getOrCreate()
silver_table="rag.silver_delta"
gold_table="rag.gold_delta"

# DBTITLE 1,Read the silver table
df = spark.read.format("delta").load(silver_table)



#This is supposed to incremental fetching new data only in the scenerio where a gold table exists 
if spark.catalog.tableExists(gold_table):
    #incremental scenerio
else:
    #full load
    for row in df.collect():
        
    chunks = chunk_document(
        content=content,
        document_id="res",
        section_id="asdfasdf"
    )

    for chunk in chunks:
        print("=" * 100)
        print(
            chunk
        )
        print(chunk["chunk_text"][:1000])